# Prepare KNN (NMSLIB) for stackoverflow_all: BM25 + IBM Model 1 fusion


(skip for now)

Run this notebook before the NMSLIB-backed evaluation notebook.

This notebook will:
1. Ensure the unified collection `stackoverflow_all` prerequisites exist.
2. Ensure BM25 and IBM Model 1 sparse-vector extractors exist.
3. Export BM25 + IBM Model 1 fusion sparse vectors for NMSLIB.
4. Create `cand_prov.json` files for both `bruteforce` and `napp` provider modes.
5. Print query-server commands for brute-force and NAPP startup.

In [3]:
import json
import os
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()
COLLECT_NAME = 'stackoverflow_all'
MODEL1_SOURCE_COLLECTION = 'stackoverflow_tran'
MODEL1_SUBDIR = 'giza'
AUTO_COPY_MODEL1_IF_MISSING = True

BM25_QUERY_FIELD = 'text'
BM25_INDEX_FIELD = 'text'
BM25_K1 = 0.4 # finetuned ones, predone
BM25_B = 0.4

MODEL1_QUERY_FIELD = 'text'
MODEL1_INDEX_FIELD = 'text'
MODEL1_GIZA_ITER_QTY = 5
MODEL1_PROB_SELF_TRAN = 0.05
MODEL1_LAMBDA = 0.1
MODEL1_MIN_MODEL1_PROB = 5e-4

# NMSLIB query server address and port.
NMSLIB_URI = 'localhost:9090'
QUERY_SERVER_PORT = 9090

# Fusion export uses an interleaved sparse vector representation.
SPARSE_INTERLEAVE = True
FUSION_EXTRACTOR_REL_PATH = 'exper_desc.best/nmslib/bm25_model1/extractor.json'
EXPORT_REL_PATH = 'nmslib/bm25_model1/export.data'
MODEL_FILE_REL = 'exper_desc.best/models/bm25_model1.model'

os.environ['COLLECT_ROOT'] = str(COLLECT_ROOT)

collect_dir = COLLECT_ROOT / COLLECT_NAME
exper_desc_dir = collect_dir / 'exper_desc'
exper_desc_best_dir = collect_dir / 'exper_desc.best'

extractor_abs_path = collect_dir / FUSION_EXTRACTOR_REL_PATH
fusion_model_abs_path = collect_dir / MODEL_FILE_REL
nmslib_bruteforce_conf_rel = 'exper_desc.best/nmslib/bruteforce/cand_prov.json'
nmslib_napp_conf_rel = 'exper_desc.best/nmslib/napp/cand_prov.json'

nmslib_bruteforce_conf_abs = collect_dir / nmslib_bruteforce_conf_rel
nmslib_napp_conf_abs = collect_dir / nmslib_napp_conf_rel

print('REPO_ROOT            =', REPO_ROOT)
print('SCRIPTS_DIR          =', SCRIPTS_DIR)
print('COLLECT_ROOT         =', COLLECT_ROOT)
print('COLLECT_NAME         =', COLLECT_NAME)
print('COLLECT_DIR          =', collect_dir)
print('Fusion extractor rel =', FUSION_EXTRACTOR_REL_PATH)
print('Fusion extractor abs =', extractor_abs_path)
print('Fusion model rel     =', MODEL_FILE_REL)
print('Fusion model abs     =', fusion_model_abs_path)
print('Bruteforce conf (rel)=', nmslib_bruteforce_conf_rel)
print('NAPP conf (rel)      =', nmslib_napp_conf_rel)
print('NMSLIB_URI           =', NMSLIB_URI)
print('EXPORT_REL_PATH      =', EXPORT_REL_PATH)
print('SPARSE_INTERLEAVE    =', SPARSE_INTERLEAVE)

REPO_ROOT            = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
SCRIPTS_DIR          = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
COLLECT_ROOT         = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
COLLECT_NAME         = stackoverflow_all
COLLECT_DIR          = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all
Fusion extractor rel = exper_desc.best/nmslib/bm25_model1/extractor.json
Fusion extractor abs = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc.best/nmslib/bm25_model1/extractor.json
Fusion model rel     = exper_desc.best/models/bm25_model1.model
Fusion model abs     = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc.best/models/bm25_model1.mod

In [4]:
required_paths = [
    collect_dir,
    collect_dir / 'input_data' / 'train' / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / 'train' / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / 'dev1' / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / 'dev1' / 'AnswerFields.jsonl',
    collect_dir / 'lucene_index',
    collect_dir / 'forward_index' / BM25_INDEX_FIELD,
    collect_dir / 'derived_data' / MODEL1_SUBDIR / MODEL1_INDEX_FIELD / 'output.t1.5.bin'
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f'Missing prerequisite: {p}')

src_model1_dir = COLLECT_ROOT / MODEL1_SOURCE_COLLECTION / 'derived_data' / MODEL1_SUBDIR
src_model1_file = src_model1_dir / MODEL1_INDEX_FIELD / 'output.t1.5.bin'
target_model1_dir = collect_dir / 'derived_data' / MODEL1_SUBDIR
target_model1_file = target_model1_dir / MODEL1_INDEX_FIELD / 'output.t1.5.bin'

print('Source Model1 dir :', src_model1_dir)
print('Source Model1 file:', src_model1_file)
print('Target Model1 dir :', target_model1_dir)
print('Target Model1 file:', target_model1_file)

if not target_model1_file.exists():
    if not AUTO_COPY_MODEL1_IF_MISSING:
        raise FileNotFoundError(
            f'Model1 missing in target collection: {target_model1_file}. '
            'Set AUTO_COPY_MODEL1_IF_MISSING=True or copy manually.'
        )
    if not src_model1_file.exists():
        raise FileNotFoundError(
            f'Source Model1 file does not exist: {src_model1_file}. '
            'Please train Model1 first or fix MODEL1_SOURCE_COLLECTION/MODEL1_INDEX_FIELD.'
        )
    target_model1_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_model1_dir, target_model1_dir, dirs_exist_ok=True)
    print('Copied Model1 directory to target collection.')

if not target_model1_file.exists():
    raise FileNotFoundError(f'Expected Model1 file after copy: {target_model1_file}')

print('All prerequisites for BM25 + IBM Model 1 fusion are present.')

Source Model1 dir : /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza
Source Model1 file: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza/text/output.t1.5.bin
Target Model1 dir : /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/derived_data/giza
Target Model1 file: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/derived_data/giza/text/output.t1.5.bin
All prerequisites for BM25 + IBM Model 1 fusion are present.


## Ensure BM25 + IBM Model 1 fusion extractor JSON exists
If the extractor file is missing, this cell generates a fusion extractor description for NMSLIB export.

In [5]:
exper_desc_best_dir.mkdir(parents=True, exist_ok=True)
extractor_abs_path.parent.mkdir(parents=True, exist_ok=True)

if not extractor_abs_path.exists():
    fusion_extractor = [
        {
            'type': 'TFIDFSimilarity',
            'params': {
                'queryFieldName': BM25_QUERY_FIELD,
                'indexFieldName': BM25_INDEX_FIELD,
                'similType': 'bm25',
                'k1': BM25_K1,
                'b': BM25_B
            }
        },
        {
            'type': 'Model1Similarity',
            'params': {
                'queryFieldName': MODEL1_QUERY_FIELD,
                'indexFieldName': MODEL1_INDEX_FIELD,
                'gizaIterQty': MODEL1_GIZA_ITER_QTY,
                'probSelfTran': MODEL1_PROB_SELF_TRAN,
                'lambda': MODEL1_LAMBDA,
                'minModel1Prob': MODEL1_MIN_MODEL1_PROB
            }
        }
    ]
    with extractor_abs_path.open('w', encoding='utf-8') as f:
        json.dump(fusion_extractor, f, indent=2)
    print('Created fusion extractor:', extractor_abs_path)

if not extractor_abs_path.exists():
    raise FileNotFoundError(f'Expected fusion extractor file was not created: {extractor_abs_path}')

print('Fusion extractor is ready:', extractor_abs_path)

Fusion extractor is ready: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc.best/nmslib/bm25_model1/extractor.json


## Export BM25 + IBM Model 1 fusion sparse vectors for NMSLIB

In [6]:
(collect_dir / 'derived_data' / 'nmslib' / 'bm25_model1').mkdir(parents=True, exist_ok=True)

if not fusion_model_abs_path.exists():
    candidates = list((collect_dir / 'exper_desc.best').rglob('*.model'))
    if not candidates:
        candidates = list(collect_dir.rglob('*.model'))
    if not candidates:
        raise FileNotFoundError(
            f'Cannot find a fusion model file under {collect_dir}. '
            'Please train the fusion model first or set MODEL_FILE_REL explicitly.'
        )
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    fusion_model_abs_path = candidates[0]
    print('Auto-selected fusion model:', fusion_model_abs_path)

cmd = [
    'bash', './export_nmslib/export_nmslib_sparse.sh',
    COLLECT_NAME,
    FUSION_EXTRACTOR_REL_PATH,
    EXPORT_REL_PATH,
    '-model_file',
    str(fusion_model_abs_path.relative_to(collect_dir)).replace('\\', '/')
]
print('Running:', ' '.join(cmd))
env = os.environ.copy()
# repo_pythonpath = str(REPO_ROOT)
# if env.get('PYTHONPATH'):
#     env['PYTHONPATH'] = repo_pythonpath + os.pathsep + env['PYTHONPATH']
# else:
#     env['PYTHONPATH'] = repo_pythonpath
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=env, text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'NMSLIB sparse export failed: {res.returncode}')

export_abs = collect_dir / 'derived_data' / EXPORT_REL_PATH
if not export_abs.exists():
    raise FileNotFoundError(f'Expected export file was not created: {export_abs}')

print('Export file ready:', export_abs)

Auto-selected fusion model: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc/models/one_feat.model
Running: bash ./export_nmslib/export_nmslib_sparse.sh stackoverflow_all exper_desc.best/nmslib/bm25_model1/extractor.json nmslib/bm25_model1/export.data -model_file exper_desc/models/one_feat.model


KeyboardInterrupt: 

## Create candidate-provider configs for notebook 05
Both configs intentionally point to the same BM25 + IBM Model 1 fusion extractor; server mode (`brute_force` vs `napp`) is selected when starting the NMSLIB query server.

In [ ]:
nmslib_bruteforce_conf_abs.parent.mkdir(parents=True, exist_ok=True)
nmslib_napp_conf_abs.parent.mkdir(parents=True, exist_ok=True)

cand_prov_payload = {
    'extrType': FUSION_EXTRACTOR_REL_PATH,
    'sparseInterleave': 'true'
}

for conf_path in [nmslib_bruteforce_conf_abs, nmslib_napp_conf_abs]:
    with conf_path.open('w', encoding='utf-8') as f:
        json.dump(cand_prov_payload, f, indent=2)
    print('Wrote:', conf_path)

print('Bruteforce conf ready:', nmslib_bruteforce_conf_abs)
print('NAPP conf ready      :', nmslib_napp_conf_abs)

## Start query server (run in a separate terminal)
Use one mode at a time, then select corresponding `PROVIDER_MODE` in notebook 05.

In [ ]:
export_abs = collect_dir / 'derived_data' / EXPORT_REL_PATH

cmd_bruteforce = f'''export COLLECT_ROOT={COLLECT_ROOT}
<path_to_query_server>/query_server \
  -p {QUERY_SERVER_PORT} \
  -s negdotprod_sparse_bin_fast \
  -i {export_abs} \
  -m brute_force'''

cmd_napp = f'''export COLLECT_ROOT={COLLECT_ROOT}
<path_to_query_server>/query_server \
  -p {QUERY_SERVER_PORT} \
  -s negdotprod_sparse_bin_fast \
  -i {export_abs} \
  -m napp'''

print('----- Brute-force server command -----')
print(cmd_bruteforce)
print()
print('----- NAPP server command -----')
print(cmd_napp)
print()
print('For notebook 05 setup:')
print("- use PROVIDER_MODE='nmslib_bruteforce' for brute-force")
print("- use PROVIDER_MODE='nmslib_napp' for NAPP")
print(f"- keep NMSLIB_URI='{NMSLIB_URI}'")
print(f"- fusion extractor: {FUSION_EXTRACTOR_REL_PATH}")
print(f"- fusion export   : {EXPORT_REL_PATH}")
print(f"- brute-force conf: {nmslib_bruteforce_conf_rel}")
print(f"- napp conf      : {nmslib_napp_conf_rel}")